# Link2iOS — The Legend of Zelda: Breath of the Wild

Notebook configurado para a pasta Google Drive compartilhada com a service account. O Colab baixa o 7z que contém o XCI do jogo base, valida o arquivo e extrai o XCI. O XCI extraído fica em `BOTW_Base_XCI/` no Drive para reutilização.

Na raiz do Drive ficam `prod.keys`, somente `update.nsp`, `dlc1.nsp` e `dlc2.nsp` como pacotes NSP; todos os demais arquivos e pastas da raiz são ignorados (mantenha também `prod.keys`). Os dois DLCs são incluídos na preparação e devem gerar imagens AOC. O arquivo 7z temporário é apagado.

O NXNative lê o XCI diretamente; não há conversão de XCI para NSP. A preparação falha se faltar qualquer pacote, se houver mais de um NSP candidato a update ou se NX2Apple não produzir AOC. O Colab prepara o jogo e o GitHub Actions/macOS finaliza a IPA NX2Apple Route C.

**Antes de executar:** o Secret `LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON` já deve estar configurado no Colab e `GOOGLE_SERVICE_ACCOUNT_JSON` no GitHub Actions. Não mova o `prod.keys` da raiz da pasta do Drive.

In [ ]:
import os
from getpass import getpass

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"

from google.colab import userdata
assert userdata.get("LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON"), "Configure o Secret LINK2IOS_GOOGLE_SERVICE_ACCOUNT_JSON no Colab."

DOWNLOAD_URL = getpass("Cole o link assinado do 7z com o XCI base (na primeira execução; pode deixar vazio se BOTW_Base_XCI já existe no Drive): ").strip()
if DOWNLOAD_URL:
    os.environ["LINK2IOS_DOWNLOAD_URL"] = DOWNLOAD_URL
else:
    os.environ.pop("LINK2IOS_DOWNLOAD_URL", None)
del DOWNLOAD_URL

print("Drive configurado. O link foi mantido apenas na memória desta sessão e não será gravado no GitHub/Drive.")

## 1. Obter o pipeline desta branch

Este clone lê os scripts da branch `Link2iOS`; nenhuma alteração local é enviada automaticamente.

In [ ]:
%cd /content
!rm -rf /content/Link2iOS-repo
!git clone --depth 1 --branch Link2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/Link2iOS-repo
!git -C /content/Link2iOS-repo rev-parse --short HEAD


## 2. Colab/Linux: download, extração e preparação BOTW

Na primeira execução, o Colab baixa o 7z pelo link assinado, testa o arquivo e extrai seu conteúdo. Exige exatamente um XCI, valida o cabeçalho `HEAD` no offset `0x100` e envia somente esse XCI para `BOTW_Base_XCI/` na pasta compartilhada. O arquivo `.7z` e a extração temporária são apagados.

O `prod.keys`, `update.nsp`, `dlc1.nsp` e `dlc2.nsp`; todos os outros arquivos e pastas são ignorados. `prod.keys` continua obrigatório são obtidos da raiz da pasta compartilhada. A chave e os três NSPs são baixados para o runtime temporário e apagados após a preparação. O XCI base extraído permanece em `BOTW_Base_XCI/` no Drive e será reutilizado nas próximas execuções; update e DLCs são baixados novamente da raiz em cada preparação.

A entrada precisa ser BOTW base XCI + update 1.9.0 + DLC1 + DLC2 (`01007EF00011E000`, `v1114112`). São exigidos exatamente um XCI, um NSP de update e dois NSPs de DLC; NX2Apple deve produzir imagens AOC.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py prepare --drive-folder-id "$DRIVE_FOLDER_ID"


## 3. Última célula: enviar o handoff e disparar somente Link2iOS

A última célula pede seu Fine-grained PAT. Ele precisa estar limitado a este repositório com **Contents: Read and write**. O token é usado apenas para criar/atualizar `handoff.json` na branch `Link2iOS`; ele não é salvo no notebook, Drive ou repositório. Esse commit é o gatilho do único workflow.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py dispatch --handoff /content/link2ios-handoff.json
